# Проверка: какие продукты попадают в вид КЕДР

Только чтение, ничего не создаёт и не меняет.

1. Какие `prod_name` / `product` есть в виде `sbx_edm_dfip.v_detail_dmkb_dkb_dmsb`.
2. Какие продукты есть у одного клиента в исходной таблице `sbx_dfip_work.detail2018`.

Логин и пароль DRP спрашиваются один раз. На скриншотах логин закрывайте `***`.

In [ ]:
from getpass import getpass

import pandas as pd
from IPython.display import display
from rail_connectors.connection import connect

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.float_format", "{:,.0f}".format)

DRP_VIEW = "sbx_edm_dfip.v_detail_dmkb_dkb_dmsb"
DRP_SOURCE = "sbx_dfip_work.detail2018"
MONTH = 202607
CHECK_INN = "1207007950"

if "drp" not in globals():
    drp = connect(
        to="DRP",
        user_params={"user_name": input("DRP login: ").strip(), "password": getpass("DRP password: ")},
    )


def drp_fetch(sql):
    with drp:
        return drp.fetch(sql)

## 1. Какие продукты есть в виде

Если в результате только «Торговый эквайринг», вид берёт только строки эквайринга, а не весь ЧОД клиента.

In [ ]:
view_products = drp_fetch(f"""
    SELECT prod_name, product, count(*) AS rows_cnt, count(DISTINCT inn) AS inn_cnt, sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm = {MONTH}
    GROUP BY prod_name, product
    ORDER BY nbi DESC
""")
display(view_products)

## 2. Один клиент: вид против исходной таблицы

Если в исходной таблице есть другие `prod_name` и сумма ближе к 64,3 млн (наша таблица), а не к 20,8 тыс. (вид), то в вид попадает только доход по эквайрингу.

Если ко второй таблице нет доступа, ячейка напишет об этом. Тогда пришлите результат первой проверки.

In [ ]:
print(f"=== Вид, ИНН {CHECK_INN}, месяц {MONTH} ===")
display(drp_fetch(f"""
    SELECT prod_name, product, item, count(*) AS rows_cnt, sum(nbi_ssp) AS nbi
    FROM {DRP_VIEW}
    WHERE yearmm = {MONTH} AND inn = '{CHECK_INN}'
    GROUP BY prod_name, product, item
    ORDER BY nbi DESC
"""))

print(f"\n=== Исходная таблица, ИНН {CHECK_INN}, месяц {MONTH}, все продукты ===")
try:
    source_inn = drp_fetch(f"""
        SELECT zo_itog, prod_name, product, count(*) AS rows_cnt, sum(nbi_ssp) AS nbi
        FROM {DRP_SOURCE}
        WHERE isactual = 1
          AND to_char(date_rep, 'yyyymm')::integer = {MONTH}
          AND inn = '{CHECK_INN}'
        GROUP BY zo_itog, prod_name, product
        ORDER BY nbi DESC
    """)
    display(source_inn)
    print("Итого nbi_ssp по клиенту:", f"{source_inn['nbi'].sum():,.0f}")
except Exception as exc:
    print("Нет доступа или ошибка:", str(exc)[:300])

## 3. Тот же клиент в озере, все продукты (Kedr.*)

Отсюда собрана наша старая таблица. Видно, из каких продуктов складывается 64,3 млн и сколько из них приходится на «Торговый эквайринг».

Логин для keytab спрашивается один раз.

In [ ]:
LAKE_VIEWS = {"dmkb": "Kedr.v_detail_dmkb", "dmsb": "Kedr.v_detail_dmsb", "dkb": "Kedr.v_detail_dkb"}

if "imp" not in globals():
    imp = connect(
        to="IMPALA",
        extra_options={"db": "sandbox_ai"},
        driver_args={"tez.queue.name": "ai"},
        kerberos={
            "keytab_path": "/home/jovyan/test_requests/tech.keytab",
            "use_credentials": True,
            "update_keytab": True,
        },
        user_params={"user_name": input("Login (keytab): ").strip()},
    )


def imp_fetch(sql):
    with imp:
        imp.execute("SET MEM_LIMIT=4g")
        return imp.fetch(sql)


for src, view in LAKE_VIEWS.items():
    print(f"\n=== {view}, ИНН {CHECK_INN}, месяц {MONTH} ===")
    try:
        cols = set(imp_fetch(f"DESCRIBE {view}").iloc[:, 0].str.lower())
        group_cols = [c for c in ("prod_name", "product", "item") if c in cols]
        group_sql = ", ".join(group_cols) if group_cols else "'-'"
        df = imp_fetch(f"""
            SELECT {group_sql}, count(*) AS rows_cnt, sum(nbi_ssp) AS nbi
            FROM {view}
            WHERE yearmm = {MONTH} AND inn = '{CHECK_INN}'
            GROUP BY {group_sql}
            ORDER BY nbi DESC
        """)
        if df.empty:
            print("Строк нет")
            continue
        display(df)
        print("Итого nbi_ssp:", f"{df['nbi'].sum():,.0f}")
    except Exception as exc:
        print("Ошибка:", str(exc)[:300])